# BiLSTM (Self-contained) - Kaggle Notebook

This notebook contains everything needed to train and evaluate a small BiLSTM-based diacritic restoration model on Kaggle, except the dataset itself (attach your dataset under `/kaggle/input/...`).

Notes: it embeds minimal data-loading, model, training and evaluation code so you can run the whole experiment from this single file.

In [ ]:
# Cell 1: Environment checks and deterministic setup
import os, sys, random, shutil
import numpy as np
import torch
print('Python', sys.version.split()[0])
print('Torch', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
# sensible thread limits for Kaggle containers
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')
torch.set_num_threads(4)
# deterministic seeds (best-effort)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print('Setup complete - PROJECT_ROOT=', PROJECT_ROOT)

In [ ]:
# Cell 2: Copy attached Kaggle dataset into ./data/ if present
# Change this path to match your attached dataset name in Kaggle UI
KAGGLE_DATASET_PATH = '/kaggle/input/yoruba-parallel'
if os.path.exists(KAGGLE_DATASET_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    for fname in os.listdir(KAGGLE_DATASET_PATH):
        src = os.path.join(KAGGLE_DATASET_PATH, fname)
        dst = os.path.join(DATA_DIR, fname)
        if os.path.isfile(src):
            print('Copying', src, '->', dst)
            shutil.copy(src, dst)
else:
    print('No attached Kaggle dataset found at', KAGGLE_DATASET_PATH, '- ensure you attached the dataset in the Notebook UI')

In [ ]:
# Cell 3: Lightweight data classes and collate function (embedded so notebook is standalone)
from typing import List, Tuple
import torch
from torch.utils.data import Dataset, DataLoader

    def __init__(self, specials=['<pad>','<unk>','<sos>','<eos>']):
        self.idx2char = list(specials)
        self.char2idx = {c:i for i,c in enumerate(self.idx2char)}
    def add_texts(self, texts: List[str]):
        for t in texts:
            for ch in t:
                if ch not in self.char2idx:
                    self.char2idx[ch] = len(self.idx2char)
                    self.idx2char.append(ch)
    def encode(self, text: str):
        return [self.char2idx.get(ch, self.char2idx['<unk>']) for ch in text]
    def decode(self, ids):
        return ''.join(self.idx2char[i] if i < len(self.idx2char) else '<unk>' for i in ids)
    @property
    def pad_index(self): return 0
    @property
    def unk_index(self): return 1
    @property
    def sos_index(self): return 2
    @property
    def eos_index(self): return 3
    def __len__(self): return len(self.idx2char)

    def __init__(self, src_texts: List[str], tgt_texts: List[str], vocab: CharVocab, max_len=128):
        assert len(src_texts) == len(tgt_texts), 'src/tgt mismatch'
        self.src = [vocab.encode(s)[:max_len] for s in src_texts]
        tgt_raw = [vocab.encode(t)[:max_len] for t in tgt_texts]
        self.dec_input = []
        self.dec_target = []
        for t in tgt_raw:
            t_trunc = t[:max_len-1]
            self.dec_input.append([vocab.sos_index] + t_trunc)
            self.dec_target.append(t_trunc + [vocab.eos_index])
        self.vocab = vocab
    def __len__(self): return len(self.src)
    def __getitem__(self, idx):
        import torch
        return torch.tensor(self.src[idx], dtype=torch.long), torch.tensor(self.dec_input[idx], dtype=torch.long), torch.tensor(self.dec_target[idx], dtype=torch.long)

            srcs, dec_ins, dec_tgts = zip(*batch)
            max_src = max(s.size(0) for s in srcs)
            max_in = max(s.size(0) for s in dec_ins)
            max_tgt = max(s.size(0) for s in dec_tgts)
            padded_src = torch.full((len(srcs), max_src), fill_value=0, dtype=torch.long)
            padded_in = torch.full((len(dec_ins), max_in), fill_value=0, dtype=torch.long)
            padded_tgt = torch.full((len(dec_tgts), max_tgt), fill_value=0, dtype=torch.long)
            for i,s in enumerate(srcs): padded_src[i,:s.size(0)] = s
            for i,s in enumerate(dec_ins): padded_in[i,:s.size(0)] = s
            for i,s in enumerate(dec_tgts): padded_tgt[i,:s.size(0)] = s
            return padded_src, padded_in, padded_tgt

In [ ]:
# Cell 4: Minimal Seq2Seq BiLSTM model with greedy decode
import torch.nn as nn
import torch

    def __init__(self, vocab_size, emb_dim=128, hid_dim=256, num_layers=1, sos_index=2, eos_index=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.encoder = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, bidirectional=True, batch_first=True)
        self.decoder = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)
        self.sos = sos_index
        self.eos = eos_index
        self.hid_dim = hid_dim
    def _combine_bidir(self, h):
        # h: num_layers*2, batch, hid -> combine pairs into num_layers, batch, hid
,